Labe mapping from eICAB segmentation masks to centerline

In [ ]:
import os
import sys
from pathlib import Path

# Add src to path
project_root = Path(os.path.abspath('')).parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from imaging import io, using


In [36]:
import nibabel as nib
import numpy as np
from scipy import ndimage as ndi

def map_labels_to_centerline_ndi(tof_path, cl_path, reg_matrix_phys, output_path, outside_label=100):
    """
    Registers TOF mask to 4D Flow space and maps labels using scipy.ndimage.
    
    Args:
        tof_path: Path to multi-label TOF mask (.nii.gz)
        cl_path: Path to 4D flow centerline mask (.nii.gz)
        reg_matrix_phys: 4x4 registration matrix (Physical Space: TOF -> 4D Flow)
        output_path: Path for output file
    """
    tof_img, tof_meta = io.imread(tof_path, axes='XYZ')
    cl_img, cl_meta = io.imread(cl_path, axes='XYZ')
    
    # print(tof_meta)
    # print(cl_meta)
    affine_tof = tof_meta['affine']
    affine_4df = cl_meta['affine']
    # T_voxel = (Inv_TOF_Affine) * (Inv_Phys_Reg) * (4DFlow_Affine)
    # Note: We invert reg_matrix_phys because we are pulling data FROM tof TO 4dflow
    phys_reg_inv = np.linalg.inv(reg_matrix_phys)
    voxel_transform = np.linalg.inv(affine_tof) @ phys_reg_inv @ affine_4df
    
    # Extract the 3x3 rotation/zoom and the translation offset for scipy
    mat_3x3 = voxel_transform[:3, :3]
    offset = voxel_transform[:3, 3]

    print("--- Resampling TOF mask to 4D Flow space ---")
    
    # 3. Apply the transform
    # order=0 is equivalent to Nearest Neighbor (crucial for discrete labels)
    registered_tof = ndi.affine_transform(
        tof_img,
        matrix=mat_3x3,
        offset=offset,
        output_shape=cl_img.shape,
        order=0,
        mode='constant',
        cval=0
    )
    io.imsave(output_path.parent / 'registered_tof.nii', registered_tof, axes='XYZ', metadata=cl_meta)

    print("--- Mapping labels to centerline ---")

    final_output = np.zeros_like(cl_img)
    
    cl_mask = cl_img > 0
    # Assign the TOF label to the centerline points
    final_output[cl_mask] = registered_tof[cl_mask]
    
    # If centerline exists but the registered TOF label is 0 (background)
    outside_mask = (cl_mask) & (registered_tof == 0)
    final_output[outside_mask] = outside_label

    # 6. Save the result
    io.imsave(output_path, final_output, axes='XYZ', metadata=cl_meta)
    print(f"Saved to {output_path}")
    print(f'Unique labels: {np.unique(final_output)}')


In [ ]:
base_path = Path('/home/imarcoss/NetVolumes/Tierra/LAB_VF-ICH/LAB/MCC LAB/_IgnacioMarcos/LabVF/PESA-Brain/RESULTS/res_GroundTruth_4DF/PESA10758400/')
output_path = base_path / 'centerline_labeled.nii'
tof_path = base_path / 'TOF_eICAB_WB.nii'
cl_path = base_path / 'branch_mask_wb.nii'
reg_matrix_phys_path = base_path / 'transform.mat'

In [24]:
matrix = np.loadtxt(reg_matrix_phys_path)

In [25]:
matrix

array([[ 9.99999523e-01, -9.59018769e-04, -1.89310391e-04,
         1.23316140e+01],
       [ 9.58991573e-04,  9.99999523e-01, -1.43656798e-04,
        -2.34011426e+00],
       [ 1.89448070e-04,  1.43475177e-04,  1.00000000e+00,
        -7.30376172e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [37]:
with using('numpy'):
    matrix = np.loadtxt(reg_matrix_phys_path)
    map_labels_to_centerline_ndi(tof_path, cl_path, matrix, output_path)

--- Resampling TOF mask to 4D Flow space ---
--- Mapping labels to centerline ---
Saved to /home/imarcoss/NetVolumes/Tierra/LAB_VF-ICH/LAB/MCC LAB/_IgnacioMarcos/LabVF/PESA-Brain/RESULTS/res_GroundTruth_4DF/PESA5745609/centerline_labeled.nii
Unique labels: [  0.   5. 100.]


In [42]:
with using('numpy'):
    registered_tof, _ = io.imread(output_path.parent / 'r_TOF_eICAB_WB.nii', axes='XYZ')
    cl_img, cl_meta = io.imread(cl_path, axes='XYZ')

    final_output = np.zeros_like(cl_img)
        
    cl_mask = cl_img > 0
    # Assign the TOF label to the centerline points
    final_output[cl_mask] = registered_tof[cl_mask]

    # If centerline exists but the registered TOF label is 0 (background)
    outside_mask = (cl_mask) & (registered_tof == 0)
    final_output[outside_mask] = 100

    # 6. Save the result
    io.imsave(output_path, final_output, axes='XYZ', metadata=cl_meta)
    print(f"Saved to {output_path}")
    print(f'Unique labels: {np.unique(final_output)}')

Saved to /home/imarcoss/NetVolumes/Tierra/LAB_VF-ICH/LAB/MCC LAB/_IgnacioMarcos/LabVF/PESA-Brain/RESULTS/res_GroundTruth_4DF/PESA5745609/centerline_labeled.nii
Unique labels: [  0.   1.   2.   3.   4.   5.   6.   7.   8.   9.  10.  11.  12.  13.
  14.  15.  16.  17.  18. 100.]
